# Attention, Encoder-Decoder Transformers, and Seq2Seq

## What We Are Going To Do

The first notebook focused on a decoder-only model that predicts the next token.
This notebook answers a different question:

**What if the model needs to read one sequence and generate a different sequence?**

That is the sequence-to-sequence setting used in tasks like:

- translation
- summarization
- paraphrasing
- structured question answering

## Big Ideas

We will connect theory to practice in three layers:

1. show why a fixed bottleneck is weak for long sequences
2. show how attention solves that problem by letting the decoder look back at the source
3. fine-tune a real encoder-decoder model on SAMSum summarization

## Learning Outcomes

By the end of the notebook, students should be able to explain:

- why attention is needed in seq2seq models
- the difference between encoder self-attention and decoder cross-attention
- how encoder and decoder hidden states interact
- how a pretrained T5 model is fine-tuned for summarization
- how cross-attention acts like task-specific memory retrieval


# Environment Setup

This notebook installs the shared environment for the whole teaching sequence.
The repository bootstrap now handles PyTorch separately so NVIDIA Windows machines
can use a CUDA-enabled wheel instead of silently falling back to a CPU-only build.

In the code cell below we:

- locate the project root
- run the repository bootstrap script
- print progress so students can see the environment being prepared


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bootstrap_script = PROJECT_ROOT / "scripts" / "bootstrap_env.py"

print(f"Project root: {PROJECT_ROOT}")
print(f"Running bootstrap script: {bootstrap_script}")
subprocess.check_call([sys.executable, str(bootstrap_script)])
print("Environment ready. If PyTorch was replaced during bootstrap, restart the kernel once before continuing.")


## Imports and Shared Helpers

We use the Hugging Face stack for the real seq2seq part of this notebook.
The theory is unchanged: we still tokenize, batch, optimize, and visualize.
The difference is that the encoder-decoder architecture is already implemented and pretrained.

In the code cell below we:

- import plotting, datasets, and Hugging Face utilities
- detect the compute device
- create a checkpoint folder for the summarization model


In [ ]:
import inspect
import random
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import DatasetDict, load_dataset
from IPython.display import display
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CHECKPOINT_DIR = PROJECT_ROOT / "artifacts" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")


## Configuration

We keep this notebook flexible for class time.
The fine-tuning section can either load a local checkpoint or run a short live fine-tuning pass.

In the code cell below we:

- choose the pretrained checkpoint
- choose how many SAMSum examples to use in class
- configure the output directory for the fine-tuned model
- display the active settings


In [ ]:
USE_EXISTING_CHECKPOINT_IF_AVAILABLE = True
MODEL_CHECKPOINT = "google-t5/t5-small"
TRAIN_SAMPLES = 2200
VALIDATION_SAMPLES = 300
TEST_SAMPLES = 120
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 64
MAX_STEPS = 220

LOCAL_OUTPUT_DIR = CHECKPOINT_DIR / "t5_samsum_demo"

seq2seq_config_df = pd.DataFrame(
    [
        ("model_checkpoint", MODEL_CHECKPOINT),
        ("train_samples", TRAIN_SAMPLES),
        ("validation_samples", VALIDATION_SAMPLES),
        ("test_samples", TEST_SAMPLES),
        ("max_source_length", MAX_SOURCE_LENGTH),
        ("max_target_length", MAX_TARGET_LENGTH),
        ("max_steps", MAX_STEPS),
        ("local_output_dir", str(LOCAL_OUTPUT_DIR)),
    ],
    columns=["setting", "value"],
)
display(seq2seq_config_df)


## Why Attention Is Needed in Seq2Seq Models

Early encoder-decoder models often tried to compress the whole input sequence into one fixed vector.
That creates a bottleneck: every part of the output has to rely on the same compressed summary.

Attention fixes that by letting each decoder step build a fresh weighted lookup over the source tokens.

In the code cell below we:

- build a toy source sequence and a toy target sequence
- compare a fixed bottleneck representation with attention-based retrieval
- visualize how different target tokens focus on different source positions


In [ ]:
source_tokens = ["alice", "ordered", "pizza", "for", "bob", "in", "chicago"]
target_tokens = ["alice", "pizza", "bob", "chicago"]

vocab = sorted(set(source_tokens + target_tokens))
token_to_id = {token: idx for idx, token in enumerate(vocab)}

source_matrix = np.eye(len(vocab))[ [token_to_id[token] for token in source_tokens] ]
query_matrix = np.eye(len(vocab))[ [token_to_id[token] for token in target_tokens] ]

raw_scores = query_matrix @ source_matrix.T
scaled_scores = raw_scores * 6.0
attention_weights = np.exp(scaled_scores) / np.exp(scaled_scores).sum(axis=1, keepdims=True)

fixed_bottleneck = source_matrix.mean(axis=0, keepdims=True)
repeated_bottleneck = np.repeat(fixed_bottleneck, len(target_tokens), axis=0)
attention_context = attention_weights @ source_matrix

fig, axes = plt.subplots(1, 3, figsize=(21, 5))
sns.heatmap(
    raw_scores,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    xticklabels=source_tokens,
    yticklabels=target_tokens,
    ax=axes[0],
)
axes[0].set_title("Raw match scores between decoder queries and source tokens")
axes[0].set_xlabel("Source positions")
axes[0].set_ylabel("Target positions")

sns.heatmap(
    attention_weights,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    xticklabels=source_tokens,
    yticklabels=target_tokens,
    ax=axes[1],
)
axes[1].set_title("Attention turns scores into a lookup distribution")
axes[1].set_xlabel("Source positions")
axes[1].set_ylabel("Target positions")

sns.heatmap(
    np.concatenate([repeated_bottleneck, attention_context], axis=1),
    cmap="magma",
    yticklabels=target_tokens,
    ax=axes[2],
)
axes[2].set_title("Left: fixed bottleneck | Right: attention-based context")
axes[2].set_xlabel("Feature dimension")
axes[2].set_ylabel("Target positions")
plt.tight_layout()
plt.show()


## Step 1: Load and Inspect SAMSum

SAMSum contains everyday messenger-style conversations paired with summaries.
That makes it ideal for teaching because students can quickly understand both the input and the target.

In the code cell below we:

- load the dataset from Hugging Face
- create small classroom subsets
- preview one dialogue-summary pair
- visualize conversation length and summary length


In [ ]:
samsum = load_dataset("knkarthick/samsum")

if "validation" not in samsum:
    split = samsum["train"].train_test_split(test_size=0.1, seed=SEED)
    samsum = DatasetDict(
        train=split["train"],
        validation=split["test"],
        test=samsum["test"] if "test" in samsum else split["test"],
    )

dialogue_column = "dialogue" if "dialogue" in samsum["train"].column_names else "dialog"
summary_column = "summary"

small_samsum = DatasetDict(
    train=samsum["train"].shuffle(seed=SEED).select(range(min(TRAIN_SAMPLES, len(samsum["train"])))),
    validation=samsum["validation"].shuffle(seed=SEED).select(range(min(VALIDATION_SAMPLES, len(samsum["validation"])))),
    test=samsum["test"].shuffle(seed=SEED).select(range(min(TEST_SAMPLES, len(samsum["test"])))),
)

example = small_samsum["train"][0]
print("Dialogue sample:")
print(example[dialogue_column])
print()
print("Reference summary:")
print(example[summary_column])

dialogue_lengths = pd.Series([len(text.split()) for text in small_samsum["train"][dialogue_column]])
summary_lengths = pd.Series([len(text.split()) for text in small_samsum["train"][summary_column]])
turn_counts = pd.Series([text.count("\n") + 1 for text in small_samsum["train"][dialogue_column]])

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.histplot(dialogue_lengths, bins=30, color="steelblue", ax=axes[0])
axes[0].set_title("Dialogue length in words")
axes[0].set_xlabel("Words")

sns.histplot(summary_lengths, bins=25, color="darkorange", ax=axes[1])
axes[1].set_title("Summary length in words")
axes[1].set_xlabel("Words")

sns.histplot(turn_counts, bins=20, color="seagreen", ax=axes[2])
axes[2].set_title("Number of turns per conversation")
axes[2].set_xlabel("Turns")
plt.tight_layout()
plt.show()


## Step 2: Tokenize Inputs and Targets

In encoder-decoder summarization we tokenize two different things:

- the source dialogue
- the target summary

T5 is also a prompted multitask model, so we prepend `summarize:` to the source text.

In the code cell below we:

- load the T5 tokenizer
- preprocess the dialogue-summary pairs
- create tokenized train, validation, and test sets
- visualize source and target length distributions


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
prefix = "summarize: "

def preprocess_batch(examples):
    model_inputs = tokenizer(
        [prefix + dialogue for dialogue in examples[dialogue_column]],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples[summary_column],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_samsum = small_samsum.map(
    preprocess_batch,
    batched=True,
    remove_columns=small_samsum["train"].column_names,
)

source_lengths = pd.Series([len(row) for row in tokenized_samsum["train"]["input_ids"]])
target_lengths = pd.Series([len(row) for row in tokenized_samsum["train"]["labels"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(source_lengths, bins=30, color="mediumpurple", ax=axes[0])
axes[0].set_title("Tokenized source lengths")
axes[0].set_xlabel("Tokens")

sns.histplot(target_lengths, bins=25, color="tomato", ax=axes[1])
axes[1].set_title("Tokenized target lengths")
axes[1].set_xlabel("Tokens")
plt.tight_layout()
plt.show()


## Step 3: Run One Forward Pass and Inspect Encoder/Decoder Shapes

The encoder reads the dialogue and produces hidden states.
The decoder reads the partly generated summary and uses cross-attention to look back into those encoder states.

In the code cell below we:

- load the pretrained T5 model
- run a forward pass on one example
- inspect the shape of logits, encoder hidden states, and attention tensors


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(DEVICE)
model.eval()

sample_dialogue = example[dialogue_column]
sample_summary = example[summary_column]

model_inputs = tokenizer(
    prefix + sample_dialogue,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SOURCE_LENGTH,
).to(DEVICE)
label_inputs = tokenizer(
    text_target=sample_summary,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
).input_ids.to(DEVICE)

with torch.no_grad():
    outputs = model(
        **model_inputs,
        labels=label_inputs,
        output_attentions=True,
        output_hidden_states=True,
        return_dict=True,
    )

shape_df = pd.DataFrame(
    [
        ("encoder hidden state", tuple(outputs.encoder_last_hidden_state.shape)),
        ("logits", tuple(outputs.logits.shape)),
        ("encoder attention layer 0", tuple(outputs.encoder_attentions[0].shape)),
        ("decoder attention layer 0", tuple(outputs.decoder_attentions[0].shape)),
        ("cross attention layer 0", tuple(outputs.cross_attentions[0].shape)),
    ],
    columns=["tensor", "shape"],
)
display(shape_df)


## Step 4: Visualize Encoder Self-Attention and Decoder Cross-Attention

This is the heart of the encoder-decoder story.

- encoder self-attention mixes information inside the source dialogue
- decoder self-attention mixes information inside the partially generated summary
- cross-attention lets each summary token decide which source tokens matter right now

In the code cell below we:

- decode tokens back into readable pieces
- visualize encoder self-attention from one layer and head
- visualize decoder cross-attention from one layer and head


In [ ]:
encoder_tokens = tokenizer.convert_ids_to_tokens(model_inputs["input_ids"][0])
decoder_input_ids = model._shift_right(label_inputs)
decoder_tokens = tokenizer.convert_ids_to_tokens(decoder_input_ids[0])

with torch.no_grad():
    attention_outputs = model(
        input_ids=model_inputs["input_ids"],
        attention_mask=model_inputs["attention_mask"],
        decoder_input_ids=decoder_input_ids,
        labels=label_inputs,
        output_attentions=True,
        return_dict=True,
    )

encoder_attention = attention_outputs.encoder_attentions[0][0, 0].detach().cpu().numpy()
cross_attention = attention_outputs.cross_attentions[0][0, 0].detach().cpu().numpy()

encoder_plot_length = min(24, len(encoder_tokens))
decoder_plot_length = min(16, len(decoder_tokens))

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.heatmap(
    encoder_attention[:encoder_plot_length, :encoder_plot_length],
    cmap="viridis",
    xticklabels=encoder_tokens[:encoder_plot_length],
    yticklabels=encoder_tokens[:encoder_plot_length],
    ax=axes[0],
)
axes[0].set_title("Encoder self-attention")
axes[0].set_xlabel("Source keys")
axes[0].set_ylabel("Source queries")

sns.heatmap(
    cross_attention[:decoder_plot_length, :encoder_plot_length],
    cmap="magma",
    xticklabels=encoder_tokens[:encoder_plot_length],
    yticklabels=decoder_tokens[:decoder_plot_length],
    ax=axes[1],
)
axes[1].set_title("Decoder cross-attention")
axes[1].set_xlabel("Source tokens the decoder can look at")
axes[1].set_ylabel("Current summary tokens")
plt.tight_layout()
plt.show()


## Step 5: Prepare the Fine-Tuning Pipeline

We now move from inspection to learning.
Fine-tuning updates pretrained weights so the model adapts to our summarization dataset.

In the code cell below we:

- build a compatibility helper for current or slightly older Hugging Face versions
- create the seq2seq data collator
- define ROUGE-based evaluation


In [ ]:
rouge = evaluate.load("rouge")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=MODEL_CHECKPOINT)

def make_seq2seq_training_args(**kwargs):
    parameters = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    if "eval_strategy" not in parameters and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    return Seq2SeqTrainingArguments(**kwargs)

def processor_argument(processor):
    parameters = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "processing_class" in parameters:
        return {"processing_class": processor}
    return {"tokenizer": processor}

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True,
    )
    prediction_lengths = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["generated_length"] = float(np.mean(prediction_lengths))
    return {key: round(value, 4) for key, value in result.items()}

print("Seq2Seq fine-tuning helpers are ready.")


## Step 6: Fine-Tune T5 or Load an Existing Checkpoint

This is the live training section of the notebook.
If a checkpoint is already present, we load it to save class time.
Otherwise we run a short fine-tuning pass and then save the result locally.

In the code cell below we:

- load or fine-tune the summarization model
- track training and evaluation metrics
- visualize the learning curve


In [ ]:
training_history = None

if LOCAL_OUTPUT_DIR.exists() and USE_EXISTING_CHECKPOINT_IF_AVAILABLE:
    seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_OUTPUT_DIR).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_OUTPUT_DIR)
    print(f"Loaded fine-tuned checkpoint from {LOCAL_OUTPUT_DIR}")
else:
    seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(DEVICE)

    training_args = make_seq2seq_training_args(
        output_dir=str(LOCAL_OUTPUT_DIR),
        max_steps=MAX_STEPS,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        logging_steps=25,
        save_total_limit=2,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        fp16=(DEVICE == "cuda"),
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=seq2seq_model,
        args=training_args,
        train_dataset=tokenized_samsum["train"],
        eval_dataset=tokenized_samsum["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        **processor_argument(tokenizer),
    )

    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)
    print(f"Saved fine-tuned checkpoint to {LOCAL_OUTPUT_DIR}")

    training_history = pd.DataFrame(trainer.state.log_history)

if training_history is not None and not training_history.empty:
    plt.figure(figsize=(10, 5))
    if "loss" in training_history:
        loss_rows = training_history.dropna(subset=["loss"])
        plt.plot(loss_rows["step"], loss_rows["loss"], label="train loss")
    if "eval_loss" in training_history:
        eval_rows = training_history.dropna(subset=["eval_loss"])
        plt.plot(eval_rows["step"], eval_rows["eval_loss"], label="eval loss")
    plt.title("T5 fine-tuning learning curve")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Step 7: Summarize Unseen Conversations

Training only matters if the model produces useful outputs on new examples.
For summarization, the easiest classroom check is to compare generated summaries against references.

In the code cell below we:

- run generation on held-out conversations
- decode predictions back into text
- compare predictions with reference summaries


In [ ]:
seq2seq_model.eval()

held_out_examples = [small_samsum["test"][idx] for idx in range(min(3, len(small_samsum["test"])))]
predictions = []

for held_out in held_out_examples:
    inputs = tokenizer(
        prefix + held_out[dialogue_column],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    ).to(DEVICE)
    generated_ids = seq2seq_model.generate(
        **inputs,
        max_new_tokens=MAX_TARGET_LENGTH,
        do_sample=False,
    )
    prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    predictions.append(
        {
            "dialogue_preview": held_out[dialogue_column][:180] + "...",
            "reference_summary": held_out[summary_column],
            "model_summary": prediction,
        }
    )

prediction_df = pd.DataFrame(predictions)
display(prediction_df)


## Step 8: Visualize Cross-Attention as Seq2Seq Memory

In decoder-only models, memory means the visible left context.
In encoder-decoder models, memory is more structured:

- the encoder stores the source sequence as hidden states
- the decoder retrieves from that memory through cross-attention

In the code cell below we:

- select one generated example
- inspect one decoder token's cross-attention distribution
- visualize which source tokens the summary token relied on


In [ ]:
attention_example = held_out_examples[0]
attention_inputs = tokenizer(
    prefix + attention_example[dialogue_column],
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SOURCE_LENGTH,
).to(DEVICE)
attention_labels = tokenizer(
    text_target=attention_example[summary_column],
    return_tensors="pt",
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
).input_ids.to(DEVICE)

with torch.no_grad():
    attention_result = seq2seq_model(
        **attention_inputs,
        labels=attention_labels,
        output_attentions=True,
        return_dict=True,
    )

cross_attention_map = attention_result.cross_attentions[0][0, 0].detach().cpu().numpy()
source_tokens = tokenizer.convert_ids_to_tokens(attention_inputs["input_ids"][0])
target_tokens = tokenizer.convert_ids_to_tokens(seq2seq_model._shift_right(attention_labels)[0])

last_target_index = min(len(target_tokens) - 1, 10)
source_plot_length = min(28, len(source_tokens))

plt.figure(figsize=(14, 5))
plt.bar(
    range(source_plot_length),
    cross_attention_map[last_target_index, :source_plot_length],
    color="crimson",
)
plt.xticks(range(source_plot_length), source_tokens[:source_plot_length], rotation=90)
plt.title(
    f"Cross-attention for target token '{target_tokens[last_target_index]}'"
)
plt.xlabel("Source token position")
plt.ylabel("Attention weight")
plt.tight_layout()
plt.show()


## Wrap-Up

This notebook made the encoder-decoder pattern concrete.
Students should now be able to separate three attention roles:

- encoder self-attention: mix information within the input
- decoder self-attention: mix information within the generated output prefix
- cross-attention: retrieve what matters from the encoded input

The final notebook moves from architecture understanding to applications:
BERT-style classification, GPT-style generation, and Hugging Face workflows.
